# 04 — Quantization Validation

Confirms INT8 ONNX + TFLite exports produce predictions consistent with the FP32 PyTorch student. We expect a small mAP drop (~1-2 pts) — anything beyond that means the calibration set was unrepresentative.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
from src.paths import EXPORTS_DIR

export_dir = EXPORTS_DIR/'rads_student_int8'
for f in sorted(export_dir.iterdir()):
    if f.is_file():
        print(f'{f.name:40s} {f.stat().st_size/1e6:7.2f} MB')

In [ ]:
# Sanity check: ONNX Runtime inference on a single image
import numpy as np, cv2, onnxruntime as ort
from src.quantize.calib_loader import _letterbox
from src.data.roboflow_pull import pull

dataset = pull()
img_path = next((dataset/'test'/'images').iterdir())
im = cv2.imread(str(img_path)); im = _letterbox(im, 640)
im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
x = np.transpose(im, (2,0,1))[None, ...]

for name in ['rads_student_int8.onnx', 'rads_student_int8.int8.onnx']:
    sess = ort.InferenceSession(str(export_dir/name), providers=['CPUExecutionProvider'])
    inp = sess.get_inputs()[0].name
    out = sess.run(None, {inp: x})
    print(f'{name}: output shapes = {[o.shape for o in out]}')

In [ ]:
# TFLite inference sanity
import tensorflow as tf
tflite_path = export_dir/'rads_student_int8.int8.tflite'
interp = tf.lite.Interpreter(model_path=str(tflite_path))
interp.allocate_tensors()
print('inputs :', interp.get_input_details())
print('outputs:', interp.get_output_details())